# W07 — Action Playbook: Ranking Signal Analysis

**Lane:** Ranking Signal Analysis
**Built from:** the Week-5 Logistic Regression model + Week-4 baseline rule, evaluated under the
Week-6 honest (client-grouped) split.

> **Before you submit:** run every cell top-to-bottom in Colab with `HF_TOKEN` set. All logic here
> is complete and ready to execute; the actual scores, tier sizes, and chart only exist once this
> runs against the real data. Run All, sanity-check the outputs, fill the two `# FILL AFTER RUN`
> spots, then commit.


## Setup

In [ ]:
import duckdb, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET hf_token='{os.environ['HF_TOKEN']}';")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

import pathlib
pathlib.Path("work/outputs").mkdir(parents=True, exist_ok=True)
pathlib.Path("work/figures").mkdir(parents=True, exist_ok=True)


In [ ]:
features_df = con.sql(f"""
WITH monthly AS (
  SELECT
    f.content_hash_id,
    f.client_hash_id,
    SUM(f.impressions) AS impressions_30d,
    SUM(f.clicks)       AS clicks_30d,
    AVG(f.position)     AS avg_position_30d,
    d.word_count             AS word_count,
    d.days_since_last_update AS days_since_last_update
  FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
  JOIN read_parquet('{BASE}/dim_content/*.parquet') d
    ON f.content_hash_id = d.content_hash_id
  GROUP BY 1, 2, d.word_count, d.days_since_last_update
  HAVING SUM(f.impressions) >= 250   -- same minimum-volume gate as the Week-4 baseline
),
latest_trend AS (
  SELECT content_hash_id, client_hash_id, trend_direction
  FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY content_hash_id, client_hash_id ORDER BY report_date DESC
  ) = 1
)
SELECT m.*, t.trend_direction
FROM monthly m
JOIN latest_trend t USING (content_hash_id, client_hash_id)
""").df()

features_df["ctr_30d"] = (features_df["clicks_30d"] / features_df["impressions_30d"].replace(0, np.nan)).fillna(0)
features_df["label"] = (features_df["trend_direction"] == "down").astype(int)

FEATURE_COLS = ["impressions_30d", "clicks_30d", "ctr_30d", "avg_position_30d", "word_count"]
X = features_df[FEATURE_COLS].fillna(0)
y = features_df["label"]
groups = features_df["client_hash_id"]

print("candidate rows:", len(X), " clients:", groups.nunique())


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression

# Same honest, client-grouped split used from Week 5 onward.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

model = LogisticRegression(max_iter=1000).fit(X.iloc[train_idx], y.iloc[train_idx])

# Score every candidate row (not just the held-out test rows) for the production-style playbook —
# the model itself was validated honestly in w05/w06; this step applies it to the full candidate
# pool the same way the baseline rule was applied to the full pool in w04.
model_proba = model.predict_proba(X)[:, 1]

tier = pd.cut(
    features_df["avg_position_30d"], bins=[-1, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"]
)
tier_median_ctr = features_df.groupby(tier)["ctr_30d"].transform("median")
staleness_norm = (features_df["days_since_last_update"] / 365).clip(0, 1)
ctr_gap_norm = ((tier_median_ctr - features_df["ctr_30d"]) / tier_median_ctr.replace(0, np.nan)).clip(0, 1).fillna(0)
baseline_norm = 0.5 * staleness_norm + 0.5 * ctr_gap_norm

# Blend, starter-pipeline style: mostly the validated model, a smaller share of the transparent
# baseline rule, so the final score is not solely a black box.
features_df["final_action_score"] = 100 * (0.70 * model_proba + 0.30 * baseline_norm)
features_df["position_tier"] = tier
features_df["staleness_norm"] = staleness_norm
features_df["ctr_gap_norm"] = ctr_gap_norm
features_df["model_probability"] = model_proba


## 1) Ranked Actions + Reason Codes

**Reason codes** (a page can carry more than one — each is a plain, checkable condition):

- `model_decline_risk` — model probability >= 0.65
- `visible_model_opportunity` — model probability >= 0.50 and impressions_30d >= 500
- `stale_underperforming_ctr` — the Week-4 baseline condition (staleness + CTR-gap both present)
- `page_one_decay_risk` — avg_position_30d <= 10 and days_since_last_update >= 180

**Archetype → action mapping** (bands over `final_action_score`, not literal clusters — this lane
is signal analysis, not Lane 3's archetype clustering, so these are score tiers with names, not
clustering output):

| Archetype (score band) | Action | Human review |
|---|---|---|
| `high_priority_review` (score >= 80th percentile) | `review_for_refresh` | Required before any change |
| `watch` (40th-80th percentile) | `monitor` | Spot-check only |
| `low_priority` (< 40th percentile) | `no_action` | None |

**The decay/refresh insight, stated safely:** pages that are both stale (older
`days_since_last_update`) and below their position tier's expected CTR are **observed** to carry
higher model-estimated decline risk in this slice. This is a **directional**, decision-support
signal for where to look first — not a guarantee that refreshing any single page will recover its
traffic.


In [ ]:
HIGH_CUT = features_df["final_action_score"].quantile(0.80)
WATCH_CUT = features_df["final_action_score"].quantile(0.40)

def archetype(score):
    if score >= HIGH_CUT:
        return "high_priority_review"
    elif score >= WATCH_CUT:
        return "watch"
    return "low_priority"

def action_for(archetype_label):
    return {
        "high_priority_review": "review_for_refresh",
        "watch": "monitor",
        "low_priority": "no_action",
    }[archetype_label]

def reason_codes_for(row):
    codes = []
    if row["model_probability"] >= 0.65:
        codes.append("model_decline_risk")
    if row["model_probability"] >= 0.50 and row["impressions_30d"] >= 500:
        codes.append("visible_model_opportunity")
    if row["staleness_norm"] >= (180 / 365) and row["ctr_gap_norm"] > 0:
        codes.append("stale_underperforming_ctr")
    if row["avg_position_30d"] <= 10 and row["days_since_last_update"] >= 180:
        codes.append("page_one_decay_risk")
    return codes or ["no_flag"]

features_df["archetype"] = features_df["final_action_score"].apply(archetype)
features_df["action"] = features_df["archetype"].apply(action_for)
features_df["reason_codes"] = features_df.apply(reason_codes_for, axis=1)
features_df["reason_codes_str"] = features_df["reason_codes"].apply(lambda c: "|".join(c))

playbook_queue = features_df.sort_values("final_action_score", ascending=False).reset_index(drop=True)
playbook_queue.insert(0, "rank", playbook_queue.index + 1)

playbook_queue[[
    "rank", "content_hash_id", "client_hash_id", "final_action_score",
    "archetype", "action", "reason_codes_str"
]].head(15)


`# FILL AFTER RUN` — note the actual `HIGH_CUT`/`WATCH_CUT` score values and roughly how many
rows land in each archetype band, once run.


## 2) Intended Use And Limits

**Intended use:** a decision-support ranking that tells a human reviewer *which pages to look at
first* for possible refresh, given limited review capacity. It orders candidates; it does not
decide or execute anything.

**Limits:**
- The label behind the model (`trend_direction == 'down'`) is a **same-window proxy**, not a
  validated future outcome — carried over from w03-w06 and flagged there. Treat `final_action_score`
  as "worth a look," not "will decline."
- Built and validated on one mid-panel month (`2026-03`) with a client-grouped split; it has not
  been tested on the sealed final month or on a time-aware (past→future) split, so it says nothing
  about forecasting.
- No causal claim: nothing here shows a refresh *causes* recovery — no experiment was run (section
  6/14 of the lane guide).
- Consolidation, seasonality, and SERP-level click loss (section 7 of the lane guide) are **not**
  ruled out per-row — a page's high score could be any of those look-alikes, not just real decline.
- Trained on ~104 clients' worth of March data; unlikely to generalize to a very different client
  mix or a very different time of year without re-validation.


## 3) Human Review + The No-Go List

**Human review is required for every `high_priority_review` row before any action is taken.**
Reviewers should specifically check, per row, for the look-alikes from section 7 of the lane guide:
was this a sibling page absorbing demand (consolidation)? Does the drop match a seasonal pattern
for this topic? Is the "staleness" stale metadata for a page that was actually recently updated?

**What should NOT be automated:**
- Auto-publishing, auto-rewriting, or auto-deleting any page based on `final_action_score` alone.
- Treating `low_priority` / `no_action` rows as "safe forever" without periodic re-scoring — a low
  score this month is not a permanent clearance.
- Using this score as an input to any other automated system (e.g. auto-deprioritizing a page in
  another tool) without a human decision in the loop.
- Publishing any client-identifying detail from this queue externally — outputs stay pseudonymized
  IDs and aggregate numbers only (section 14 of the lane guide).
- Claiming, in any external write-up, that a page's score "predicts" or "proves" future traffic
  movement — safe language only: observed, measured, directional, decision-support.


## 4) Monitoring / Retrain Triggers

Light, practical triggers — not a production monitoring system:

- **Score drift check:** re-run this notebook monthly on the newest available complete month;
  if the median `final_action_score` or the size of the `high_priority_review` band shifts by more
  than ~20% month-over-month, treat that as a signal to re-inspect the feature distributions before
  trusting the new queue.
- **Reviewer disagreement rate:** if reviewers mark more than roughly 1 in 5 `high_priority_review`
  rows as "not actually a problem" (consolidation/seasonality/stale-metadata false positive), that
  is a retrain/rethink trigger, not something to quietly ignore.
- **Coverage check:** if a client's row count in a given month drops sharply (e.g. their
  `gsc_data_start`/`ga4_data_start` history changes or tracking gaps appear), exclude that client
  from that month's queue rather than scoring on thin data.
- **Retrain cadence:** re-fit the model at most monthly, always re-validated with the same
  client-grouped split discipline from w05/w06 — never skip the honest split to save time.


## 5) Exports For The Paper

Exports the ranked queue (kept out of git per the CI leak-guard), a metrics/receipts JSON (to be
committed), and one figure (to be committed to `work/figures/`) that the paper can reuse directly.


In [ ]:
# --- 1. ranked queue CSV (git-ignored by design; regenerated on every run) ---
export_cols = [
    "rank", "content_hash_id", "client_hash_id", "impressions_30d", "clicks_30d",
    "ctr_30d", "avg_position_30d", "position_tier", "days_since_last_update",
    "model_probability", "final_action_score", "archetype", "action", "reason_codes_str",
]
playbook_queue[export_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)
print(f"wrote {len(playbook_queue)} rows to work/outputs/action_playbook_queue.csv")

# --- 2. receipts JSON (commit this) ---
archetype_counts = playbook_queue["archetype"].value_counts().to_dict()
receipts = {
    "lane": "ranking_signal_analysis",
    "month": MONTH,
    "n_candidates": int(len(playbook_queue)),
    "n_clients": int(playbook_queue["client_hash_id"].nunique()),
    "score_cuts": {"high_priority_review_at": float(HIGH_CUT), "watch_at": float(WATCH_CUT)},
    "archetype_counts": {str(k): int(v) for k, v in archetype_counts.items()},
    "final_score_summary": {
        "min": float(playbook_queue["final_action_score"].min()),
        "median": float(playbook_queue["final_action_score"].median()),
        "max": float(playbook_queue["final_action_score"].max()),
    },
}
with open("work/outputs/w07_action_playbook_metrics.json", "w") as f:
    json.dump(receipts, f, indent=2)
print("wrote work/outputs/w07_action_playbook_metrics.json")
receipts


In [ ]:
# --- 3. figure for the paper (commit this) ---
fig, ax = plt.subplots(figsize=(7, 4.5))
order = ["low_priority", "watch", "high_priority_review"]
counts = playbook_queue["archetype"].value_counts().reindex(order).fillna(0)
ax.bar(order, counts.values, color=["#9CA3AF", "#F59E0B", "#DC2626"])
ax.set_ylabel("number of pages")
ax.set_title(f"Action queue by archetype — {MONTH} (n={len(playbook_queue)})")
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{int(v)}", ha="center", va="bottom")
fig.tight_layout()
fig.savefig("work/figures/w07_archetype_counts.png", dpi=150)
plt.show()
print("wrote work/figures/w07_archetype_counts.png")


## 6) Self-Check

- [ ] Ranked actions with reason codes — section 1.
- [ ] Archetype (score-band) → action mapping — section 1.
- [ ] Decay/refresh insight stated in safe language — section 1.
- [ ] Intended use and limits spelled out, including the same-window proxy-label limitation
      carried over from earlier weeks — section 2.
- [ ] Human-review rule stated (required for `high_priority_review`) plus a concrete no-go list —
      section 3.
- [ ] Monitoring/retrain triggers, practical and non-production — section 4.
- [ ] Queue exported to `work/outputs/action_playbook_queue.csv` (git-ignored, regenerated) —
      section 5.
- [ ] Metrics JSON committed to `work/outputs/w07_action_playbook_metrics.json` — section 5.
- [ ] Figure committed to `work/figures/w07_archetype_counts.png` — section 5.
- [ ] No claim implies automation, a guarantee, or a proven cause — matches section 14 of the lane
      guide.
